# Hotel Booking Cancellation Prediction using Deep Learning

## Business Problem
Hotel booking cancellations create major financial and operational challenges for hotels. 
Late cancellations can lead to revenue loss, inefficient room allocation, and inaccurate demand forecasting.

This project uses Deep Learning techniques to predict whether a hotel booking will be canceled based on customer and reservation information.

---

## Project Goals
- Predict booking cancellations
- Reduce financial losses
- Improve hotel resource planning
- Explore neural network performance on tabular business data

---

## Technologies Used
- Python
- Pandas
- Scikit-learn
- TensorFlow / Keras
- Matplotlib

---

## Dataset
Hotel Booking Demand Dataset

# Importing Libraries

In this section, we import all required libraries for:
- data processing
- visualization
- preprocessing
- machine learning pipelines
- deep learning

In [ ]:
# Setup plotting
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

plt.style.use('seaborn-v0_8-whitegrid')

# Set Matplotlib defaults
plt.rc('figure', autolayout=True)
plt.rc(
    'axes',
    labelweight='bold',
    labelsize='large',
    titleweight='bold',
    titlesize=18,
    titlepad=10
)

plt.rc('animation', html='html5')

# Loading the Dataset

We load the hotel booking dataset and separate:
- input features (X)
- target variable (y)

Target:
- `is_canceled`
    - 1 → booking canceled
    - 0 → booking not canceled

In [ ]:
hotel = pd.read_csv('hotel.csv.zip')

X = hotel.copy()
y = X.pop('is_canceled')

#The dataset contains month names as text values.
X['arrival_date_month'] = \
    X['arrival_date_month'].map(
        {'January':1, 'February': 2, 'March':3,
         'April':4, 'May':5, 'June':6, 'July':7,
         'August':8, 'September':9, 'October':10,
         'November':11, 'December':12}
    )

# Selecting Numerical and Categorical Features

The dataset contains:
- numerical features
- categorical features

We separate them because each type requires different preprocessing techniques.

In [ ]:

features_num = [
    "lead_time", "arrival_date_week_number",
    "arrival_date_day_of_month", "stays_in_weekend_nights",
    "stays_in_week_nights", "adults", "children", "babies",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "required_car_parking_spaces",
    "total_of_special_requests", "adr",
]
features_cat = [
    "hotel", "arrival_date_month", "meal",
    "market_segment", "distribution_channel",
    "reserved_room_type", "deposit_type", "customer_type",
]

# Data Preprocessing Pipeline

We create preprocessing pipelines to:
- handle missing values
- scale numerical data
- encode categorical variables

This ensures the data is ready for neural network training.

In [ ]:
transformer_num = make_pipeline(
    SimpleImputer(strategy="constant"), # there are a few missing values
    StandardScaler(),
)
transformer_cat = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="NA"),
    OneHotEncoder(handle_unknown='ignore'),
)

preprocessor = make_column_transformer(
    (transformer_num, features_num),
    (transformer_cat, features_cat),
)

# stratify - make sure classes are evenlly represented across splits
# The dataset is divided into:
#- training data
#- validation data
X_train, X_valid, y_train, y_valid = \
    train_test_split(X, y, stratify=y, train_size=0.75)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)

input_shape = [X_train.shape[1]]

# Building the Neural Network

We create a deep neural network using:
- Dense layers
- Batch Normalization
- Dropout regularization

The model predicts the probability of hotel booking cancellation.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers


# Define model
model = keras.Sequential([

    # Input normalization
    layers.BatchNormalization(input_shape=input_shape),

    # First hidden block
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Second hidden block
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    # Output layer
    layers.Dense(1, activation='sigmoid'),
])

# Model Compilation

The model is configured with:
- Adam optimizer
- Binary Crossentropy loss
- Binary Accuracy metric

These settings are appropriate for binary classification problems.

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)

# Training the Model

The model is trained using:
- mini-batch gradient descent
- validation monitoring
- early stopping to reduce overfitting

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=5,
    min_delta=0.001,
    restore_best_weights=True,
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=200,
    callbacks=[early_stopping],
)
# Evaluating Model Performance

#We visualize: training loss- validation loss- training accuracy- validation accuracy
#These learning curves help evaluate: convergence- model stability- overfitting behavior

history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot(title="Cross-entropy")
history_df.loc[:, ['binary_accuracy', 'val_binary_accuracy']].plot(title="Accuracy")